In [ ]:
#@title **Required settings:**
!nvidia-smi

# @markdown #### **Youtube video**
yt_url = "https://youtu.be/VscdPA6sUkc"  # @param {type:"string"}

# @markdown Optionally enter a media path already in this runtime to skip YouTube.
media_path = ""  # @param {type:"string"}

# @markdown #### **Initial prompt**
# @markdown Prompts can be very helpful for correcting specific words or acronyms that the model often misrecognizes in the audio.
prompt = "Game development talk"  # @param {type:"string"}

# @markdown #### Model
model_size = "large-v2"  # @param ["base", "base.en", "small", "small.en","medium", "medium.en", "large-v1","large-v2","large-v3"]

# @markdown #### Language
language = "en" # @param ["auto", "en", "zh", "de", "es", "ru", "ko", "fr", "ja", "pt", "tr", "pl", "ca", "nl", "ar", "sv", "it", "id", "hi", "fi", "vi", "he", "uk", "el", "ms", "cs", "ro", "da", "hu", "ta", "no", "th", "ur", "hr", "bg", "lt", "la", "mi", "ml", "cy", "sk", "te", "fa", "lv", "bn", "sr", "az", "sl", "kn", "et", "mk", "br", "eu", "is", "hy", "ne", "mn", "bs", "kk", "sq", "sw", "gl", "mr", "pa", "si", "km", "sn", "yo", "so", "af", "oc", "ka", "be", "tg", "sd", "gu", "am", "yi", "lo", "uz", "fo", "ht", "ps", "tk", "nn", "mt", "sa", "lb", "my", "bo", "tl", "mg", "as", "tt", "haw", "ln", "ha", "ba", "jw", "su"]

# @markdown #### Filename Type
# @markdown Use YouTube title as file name by default
filename_type = "id"  # @param ["title", "id"]

# @markdown #### Assign speaker labels
# @markdown For speaker labels, grant this notebook access to the HF_TOKEN Colab secret.
assign_speaker_lable = False # @param {type:"boolean"}

# @markdown #### Align whisper output
align_whisper_output = True # @param {type:"boolean"}

In [ ]:
#@title Install WhisperX and yt-dlp
import subprocess
import sys
from pathlib import Path

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv', 'yt-dlp[default]'], check=True)
wx_env = Path('/content/whisperx-venv')
wx_python = wx_env / 'bin/python'
uv = [sys.executable, '-m', 'uv']
if not wx_python.exists():
    subprocess.run(uv + ['venv', '--python', '3.11', str(wx_env)], check=True)
subprocess.run(uv + ['pip', 'install', '--python', str(wx_python), 'whisperx==3.8.6'], check=True)
subprocess.run([str(wx_env / 'bin/whisperx'), '--help'], stdout=subprocess.DEVNULL, check=True)
print('WhisperX is ready.')

In [ ]:
#@title Prepare audio or video
from pathlib import Path
import subprocess
import sys
import tempfile
from google.colab import files

video_path_local_list = []
if media_path.strip():
    source = Path(media_path.strip()).expanduser().resolve()
    if not source.is_file():
        raise FileNotFoundError(source)
    video_path_local_list.append(source)
else:
    if yt_url.strip():
        with tempfile.TemporaryDirectory() as work:
            manifest = Path(work) / 'downloaded.txt'
            template = '%(id)s' if filename_type == 'id' else '%(title).160B [%(id)s]'
            cmd = [sys.executable, '-m', 'yt_dlp', '--no-simulate', '--newline',
                   '-f', 'bestaudio/best', '--restrict-filenames', '-o',
                   '/content/media/' + template + '.%(ext)s', '--print-to-file',
                   'after_move:filepath', str(manifest), '--', yt_url.strip()]
            download = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
            if manifest.exists():
                video_path_local_list.extend(Path(p) for p in manifest.read_text().splitlines() if Path(p).is_file())
            if download.returncode:
                print(download.stdout[-2000:])
                print('YouTube download failed or was incomplete.')
    if not video_path_local_list:
        print('Choose a local audio or video file to transcribe, or set media_path and rerun this cell.')
        video_path_local_list = [Path(p).resolve() for p in files.upload().keys()]
if not video_path_local_list:
    raise ValueError('No media selected.')
print('Ready:', *(p.name for p in video_path_local_list), sep='\n')

In [ ]:
#@title Transcribe with WhisperX
import os
import subprocess
import tempfile
from pathlib import Path
from google.colab import userdata

srt_paths = []
if not video_path_local_list:
    raise ValueError('Run the media cell and select at least one input.')
options = ['--model', model_size, '--output_format', 'srt', '--batch_size', '4']
if language != 'auto':
    options += ['--language', language]
if prompt.strip():
    options += ['--initial_prompt', prompt]
if not align_whisper_output:
    options += ['--no_align']
env = os.environ.copy()
env['MPLBACKEND'] = 'Agg'
if assign_speaker_lable:
    env['HF_TOKEN'] = userdata.get('HF_TOKEN')
    options.append('--diarize')
output_dir = Path('/content/subtitles')
output_dir.mkdir(exist_ok=True)
for source in video_path_local_list:
    with tempfile.TemporaryDirectory(dir=output_dir) as work:
        command = [str(wx_env / 'bin/whisperx'), str(source), '--output_dir', work, *options]
        print('Transcribing', source.name, flush=True)
        last_message = ""
        with subprocess.Popen(command, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as process:
            for line in process.stdout:
                if line.strip():
                    last_message = line.strip()
                print(line, end='', flush=True)
            status = process.wait()
        if status:
            raise RuntimeError(f'{source.name}: WhisperX exited with {status}. {last_message}')
        result = Path(work) / (source.stem + '.srt')
        destination = output_dir / result.name
        result.replace(destination)
        srt_paths.append(destination)
        print('Saved', destination)

In [ ]:
#@title Download subtitles
from google.colab import files


if not srt_paths:
    raise RuntimeError('Run transcription first to generate subtitles.')
for subtitle in srt_paths:
    files.download(str(subtitle))